# Lakehouse Federation

## Connection to the Neon database
Password was set up using Databricks CLI.

In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS neon_pg TYPE postgresql
OPTIONS (
    host    'ep-nameless-brook-b26w7gmr-pooler.c-6.eu-central-1.aws.neon.tech',
    port    '5432',
    user    'neondb_owner',
    password secret('neon', 'pg_password'),
    trustServerCertificate 'true'
);

## Creating a foreign catalog to mirror the Neon database into Unity Catalog

In [0]:
%sql
CREATE FOREIGN CATALOG IF NOT EXISTS neon USING CONNECTION neon_pg
OPTIONS (database 'neondb');

## Sanity check

In [0]:
%sql
SELECT * FROM neon.public.dc_dim ORDER BY it_power_mw DESC;

## Joining the external table to the Delta Table
The dc_dim table is also used in the gold layer of the Entsoe project as an additional dimension. Here, the join is done for the purposes of the Lab.

In [0]:
# Configuration
from pyspark.sql import functions as F, Window
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("gold_schema", "gold", ["gold", "gabrielajaniszews786_gold"], "Gold schema")
CATALOG       = dbutils.widgets.get("catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")


In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.consumption_hourly LIMIT 10"))

In [0]:
# Joined table with the latest consumption data and data center metadata
joined_table = spark.sql(f"""
SELECT
cons.site_id,
cons.bidding_zone,
cons.date,
cons.hour,
cons.avg_consumption_kwh AS avg_consumption_kwh,
CASE
WHEN cons.avg_consumption_kwh IS NOT NULL THEN true
ELSE FALSE
END AS is_energy_consumed,
cons.avg_power_kw AS avg_power_kw,
cons.avg_pue AS avg_pue,
cons.cost_per_hour AS cost_per_hour,
dc.operator,
dc.city,
dc.longitude,
dc.latitude,
dc.it_power_mw,
dc.power_density_kw_m2
FROM {CATALOG}.{GOLD_SCHEMA}.consumption_hourly AS cons
LEFT JOIN neon.public.dc_dim AS dc
ON cons.site_id = dc.dc_id
QUALIFY ROW_NUMBER() OVER (PARTITION BY cons.site_id ORDER BY cons.date DESC, cons.hour DESC) = 1;""")

display(joined_table)


## Change Data Capture

In [0]:
spark.sql(f"""CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.cdc_table 
          TBLPROPERTIES(delta.enableChangeDataFeed = true) 
          AS SELECT * FROM neon.public.dc_dim""")

display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{GOLD_SCHEMA}.cdc_table"))

The initial load version is 0.